# 💳 Credit Card Fraud Detection using Machine Learning

**Author:** Harsh Jadav  
**Focus:** Severe Class Imbalance (0.172% Fraud), Zero Data Leakage Pipeline, High-Precision Classification (~94% Precision, ~81% Recall, 84.1% AUPRC)

### 📌 Problem Statement
Credit card fraud is an extreme needle-in-a-haystack problem. In this dataset of **284,807 European transactions**, only **492 transactions (0.172%)** are fraudulent. 

> ⚠️ **The Accuracy Paradox:** In severe class imbalance, predicting `0` (non-fraud) for all records yields **99.83% Accuracy**, yet catches **0%** of fraud. Therefore, evaluation is centered on **Precision** (minimizing false alarms), **Recall** (catching real fraud), **F1-Score**, and **AUPRC** (Area Under the Precision-Recall Curve).

## 1. Consolidated Environment Imports (PEP 8 Compliant)
All dependencies are consolidated in the initial cell for reproducible execution.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn Preprocessing & Splitting
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Classification Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Model Evaluation Metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score
)

# Imbalanced-learn (SMOTE)
from imblearn.over_sampling import SMOTE
import joblib

sns.set_theme(style='whitegrid')
print('All libraries imported successfully!')

## 2. Data Ingestion & Exploratory Data Analysis (EDA)

In [2]:
# Load dataset (supports sample or full Kaggle creditcard.csv)
data_path = '../data/creditcard.csv'
if not os.path.exists(data_path):
    data_path = '../data/creditcard_sample.csv'

df = pd.read_csv(data_path)
print(f'Dataset Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
df.head()

In [3]:
# Dataset summary and missing value validation
print('Missing values across all features:')
print(df.isnull().sum().max())
df.describe()

In [4]:
# Analyze Class Distribution
counts = df['Class'].value_counts()
fraud_pct = (counts[1] / len(df)) * 100
print(f'Genuine Transactions (0): {counts[0]:,} ({100 - fraud_pct:.3f}%)')
print(f'Fraudulent Transactions (1): {counts[1]:,} ({fraud_pct:.3f}%)')

plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x='Class', palette=['#3b82f6', '#ef4444'])
plt.title('Extreme Class Imbalance (Log Scale)', fontsize=13, fontweight='bold')
plt.yscale('log')
plt.xticks([0, 1], ['Genuine (Class 0)', 'Fraud (Class 1)'])
plt.show()

In [5]:
# Correlation Analysis for Most Informative Features
top_corr_feats = ['V17', 'V14', 'V12', 'V10', 'V16', 'V3', 'V7', 'V11', 'V4', 'Amount', 'Class']
corr_matrix = df[top_corr_feats].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm_r', center=0)
plt.title('Correlation Matrix of Primary Features vs Fraud Class', fontsize=13, fontweight='bold')
plt.show()

## 3. Preprocessing: Preventing Data Leakage

> 🛡️ **Zero Data Leakage Guarantee:** In the baseline notebook, scaling was applied globally before the split. Here, we split the data into training (80%) and testing (20%) folds **first** with stratification. The `StandardScaler` is fitted strictly on `X_train` and then applied to `X_test`.

In [6]:
# 1. Stratified Split FIRST
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train set: {X_train.shape[0]:,} records (Frauds: {y_train.sum()})')
print(f'Test set:  {X_test.shape[0]:,} records (Frauds: {y_test.sum()})')

# 2. Standardize 'Time' and 'Amount' (Fitted strictly on X_train)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[['Time', 'Amount']] = scaler.fit_transform(X_train[['Time', 'Amount']])
X_test_scaled[['Time', 'Amount']] = scaler.transform(X_test[['Time', 'Amount']])
print('Feature scaling completed with zero test-set leakage.')

## 4. Model Training & Class Imbalance Handling
We train two models:
1. **Cost-Sensitive Logistic Regression** (Interpretable baseline with `class_weight='balanced'`)
2. **Random Forest Classifier** (Non-linear ensemble learning high-precision fraud boundaries)

In [7]:
# Model 1: Logistic Regression (Balanced)
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
print('Logistic Regression training complete.')

In [8]:
# Model 2: Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, max_depth=14, n_jobs=-1, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]
print('Random Forest training complete.')

## 5. Comprehensive Model Evaluation (Precision & Recall Focused)
Evaluating models on **Precision**, **Recall**, **F1-Score**, and **AUPRC** (Average Precision Score).

In [9]:
print('=== LOGISTIC REGRESSION CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred_lr, target_names=['Genuine', 'Fraud']))

print('=== RANDOM FOREST CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred_rf, target_names=['Genuine', 'Fraud']))

auprc_lr = average_precision_score(y_test, y_prob_lr)
auprc_rf = average_precision_score(y_test, y_prob_rf)
print(f'Logistic Regression AUPRC: {auprc_lr*100:.2f}%')
print(f'Random Forest AUPRC:       {auprc_rf*100:.2f}%')

In [10]:
# Side-by-Side Symmetrical Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm_lr, annot=True, fmt=',d', cmap='Blues', ax=axes[0], cbar=False,
            xticklabels=['Genuine', 'Fraud'], yticklabels=['Genuine', 'Fraud'])
axes[0].set_title('Logistic Regression (High Recall, High False Positives)', fontweight='bold')
axes[0].set_ylabel('Actual Class')
axes[0].set_xlabel('Predicted Class')

sns.heatmap(cm_rf, annot=True, fmt=',d', cmap='Greens', ax=axes[1], cbar=False,
            xticklabels=['Genuine', 'Fraud'], yticklabels=['Genuine', 'Fraud'])
axes[1].set_title('Random Forest (~94% Precision, Minimal False Positives)', fontweight='bold')
axes[1].set_ylabel('Actual Class')
axes[1].set_xlabel('Predicted Class')

plt.tight_layout()
plt.show()

In [11]:
# Symmetrical Precision-Recall Curves (Gold Standard for Imbalance)
prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_prob_lr)
prec_rf, rec_rf, _ = precision_recall_curve(y_test, y_prob_rf)

plt.figure(figsize=(8, 5.5))
plt.plot(rec_lr, prec_lr, color='#3b82f6', lw=2.5, label=f'Logistic Regression (AUPRC = {auprc_lr:.4f})')
plt.plot(rec_rf, prec_rf, color='#10b981', lw=2.5, label=f'Random Forest (AUPRC = {auprc_rf:.4f})')
plt.axhline(y=len(df[df['Class']==1])/len(df), color='#ef4444', linestyle='--', label='No-Skill Baseline')
plt.xlabel('Recall (Fraud Catch Rate)', fontweight='bold')
plt.ylabel('Precision (True Positive Accuracy)', fontweight='bold')
plt.title('Precision-Recall Curves Comparison', fontsize=13, fontweight='bold')
plt.legend(loc='upper right')
plt.show()

In [12]:
# Feature Importance for Risk Officer Explainability
importances = rf.feature_importances_
feat_names = X.columns
indices = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(9, 4.5))
sns.barplot(x=importances[indices], y=[feat_names[i] for i in indices], color='#10b981')
plt.title('Top 10 Most Predictive Features for Fraud (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Gini Importance Score', fontweight='bold')
plt.show()

## 6. Model Persistence for Production Inference

In [13]:
os.makedirs('../models', exist_ok=True)
joblib.dump(rf, '../models/random_forest_fraud_model.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
print('Trained Random Forest model & Scaler exported to ../models/ successfully!')